# Imports + paths + load data

In [20]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import joblib


In [21]:
# --- Paths ---
cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
elif (cwd / "notebooks").exists():
    project_root = cwd
else:
    project_root = cwd

data_path = project_root / "data" / "processed" / "original_data.csv"

final_dir = project_root / "data" / "final"
final_dir.mkdir(parents=True, exist_ok=True)

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# --- Load ---
df = pd.read_csv(data_path)
print("Loaded:", data_path.resolve())
print("Shape:", df.shape)

# Explicit target
target_col = "Diabetes_binary"
assert target_col in df.columns, f"{target_col} not found!"

df.head()

Loaded: C:\Users\Lu\OneDrive\ToU\chl_Classification\data\processed\original_data.csv
Shape: (253680, 22)


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


# Train/test split (Stratified because of class imbalance)

In [22]:
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

counts = y.value_counts()
props = y.value_counts(normalize=True)
display(counts)
display(props.rename("proportion"))


X shape: (253680, 21)
y shape: (253680,)


Diabetes_binary
0    218334
1     35346
Name: count, dtype: int64

Diabetes_binary
0    0.860667
1    0.139333
Name: proportion, dtype: float64

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# sanity check: stratification preserved
display(y_train.value_counts(normalize=True).rename("train_proportion"))
display(y_test.value_counts(normalize=True).rename("test_proportion"))

# Save CSVs
X_train.to_csv(final_dir / "X_train.csv", index=False)
X_test.to_csv(final_dir / "X_test.csv", index=False)
y_train.to_csv(final_dir / "y_train.csv", index=False)
y_test.to_csv(final_dir / "y_test.csv", index=False)

print("Saved to:", final_dir.resolve())


Train shape: (202944, 21) Test shape: (50736, 21)


Diabetes_binary
0    0.860666
1    0.139334
Name: train_proportion, dtype: float64

Diabetes_binary
0    0.860671
1    0.139329
Name: test_proportion, dtype: float64

Saved to: C:\Users\Lu\OneDrive\ToU\chl_Classification\data\final


# Build preprocessing pipeline

Identify discrete vs continuous-ish (using cutoff = 5) = Same logic as in EDA

In [24]:
unique_counts = X.nunique().sort_values()

discrete_cols = unique_counts[unique_counts <= 5].index.tolist()
continuous_cols = unique_counts[unique_counts > 5].index.tolist()

print("Discrete cols (<=5 unique):", len(discrete_cols))
print(discrete_cols)

print("\nContinuous-ish cols (>5 unique):", len(continuous_cols))
print(continuous_cols)


Discrete cols (<=5 unique): 15
['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'HeartDiseaseorAttack', 'Stroke', 'PhysActivity', 'Fruits', 'NoDocbcCost', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'Sex', 'DiffWalk', 'GenHlth']

Continuous-ish cols (>5 unique): 6
['Education', 'Income', 'Age', 'MentHlth', 'PhysHlth', 'BMI']


Pipeline for continuous-ish: impute then scale

In [25]:
continuous_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline for discrete: impute most frequent (keeps 0/1 etc. intact)
discrete_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipe, continuous_cols),
        ("discrete", discrete_pipe, discrete_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Fit on TRAIN only
preprocessor.fit(X_train)

print("Preprocessor fitted on X_train.")


Preprocessor fitted on X_train.


Transform train/test

In [26]:
X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)

print("Transformed train shape:", X_train_t.shape)
print("Transformed test shape:", X_test_t.shape)

Transformed train shape: (202944, 21)
Transformed test shape: (50736, 21)


# Save preprocessor

In [27]:
preprocessor_path = models_dir / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)

print("Saved preprocessor:", preprocessor_path.resolve())


Saved preprocessor: C:\Users\Lu\OneDrive\ToU\chl_Classification\models\preprocessor.joblib
